# SpeedMart YOLO training (F13)

Trains YOLO11n on the Roboflow export of the shelf photos. Full steps: `training/TRAINING.md`.

1. **Runtime > Change runtime type > T4 GPU** (or any GPU), then **Runtime > Run all**.
2. Cell 3 asks for the Roboflow zip (export format **YOLOv8**). Pick the file on your laptop.
3. About 10 to 20 minutes later cell 7 prints mAP50 per class (target: at least 0.90 for every class) and
   cell 9 downloads `speedmart_yolo.pt`. Put it in the repo at `models/speedmart_yolo.pt`.

In [ ]:
# 1. GPU check: should print a Tesla T4 (or similar). If it errors, switch the runtime to GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install Ultralytics (YOLO11 needs >= 8.3)
!pip install -q "ultralytics>=8.3"
import ultralytics
ultralytics.checks()

In [ ]:
# 3. Upload the Roboflow export zip and unzip it to /content/dataset
#    (Alternative: skip the upload by copying Roboflow's "download code" snippet into a new cell and
#    unzipping to /content/dataset. Never paste your Roboflow API key into a notebook you share.)
import glob, os, shutil, zipfile
from google.colab import files

DATASET = "/content/dataset"
uploaded = files.upload()
zips = [name for name in uploaded if name.lower().endswith(".zip")]
assert zips, "Upload the .zip that Roboflow exported (format YOLOv8)."
shutil.rmtree(DATASET, ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall(DATASET)
# some exports wrap everything in one extra folder
found = glob.glob(f"{DATASET}/**/data.yaml", recursive=True)
assert found, "No data.yaml in the zip. Export again with format YOLOv8."
DATASET = os.path.dirname(sorted(found, key=len)[0])
print("dataset at", DATASET)
!ls {DATASET}

In [ ]:
# 4. Point data.yaml at this folder and check the class names.
#    They must equal catalog.json yolo_class values (order does not matter).
import yaml

EXPECTED = {"chips", "energy_drink", "hydration_drink", "vegan_snack", "water"}  # catalog.json yolo_class; update if the catalog changes

path = os.path.join(DATASET, "data.yaml")
data = yaml.safe_load(open(path))
data["path"] = DATASET
data["train"] = "train/images"
data["val"] = "valid/images" if os.path.isdir(os.path.join(DATASET, "valid")) else "val/images"
if os.path.isdir(os.path.join(DATASET, "test", "images")):
    data["test"] = "test/images"
else:
    data.pop("test", None)
yaml.safe_dump(data, open(path, "w"), sort_keys=False)
names = data["names"] if isinstance(data["names"], list) else [data["names"][k] for k in sorted(data["names"])]
print("classes:", names)
for split in ("train", "val"):
    folder = os.path.join(DATASET, data[split])
    print(f"{split}: {len(os.listdir(folder))} images")
missing, extra = EXPECTED - set(names), set(names) - EXPECTED
assert not missing and not extra, f"Class names do not match catalog.json. Missing {missing}, unexpected {extra}. Rename classes in Roboflow and export again."
print("class names OK")

In [ ]:
# 5. Train. patience=20 stops early when validation stops improving.
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(data=path, epochs=80, imgsz=640, batch=16, patience=20, project="/content/runs", name="speedmart", exist_ok=True)
RUN_DIR = str(results.save_dir) if results is not None and hasattr(results, "save_dir") else "/content/runs/speedmart"
BEST = os.path.join(RUN_DIR, "weights", "best.pt")
print("best weights:", BEST)

In [ ]:
# 6. Training curves (loss should go down, mAP50 up and flatten)
from IPython.display import Image, display
for name in ("results.png", "confusion_matrix.png"):
    p = os.path.join(RUN_DIR, name)
    if os.path.exists(p):
        display(Image(filename=p, width=900))

In [ ]:
# 7. Validation metrics per class on the valid split. Target: mAP50 >= 0.90 for EVERY class.
best = YOLO(BEST)
metrics = best.val(data=path, imgsz=640, split="val")
box = metrics.box
print()
print(f"{'class':<18}{'P':>8}{'R':>8}{'mAP50':>8}{'mAP50-95':>10}")
all_ok = True
for i, cls in enumerate(box.ap_class_index):
    name = metrics.names[int(cls)]
    p, r, ap50, ap = box.class_result(i)
    ok = ap50 >= 0.90
    all_ok &= ok
    print(f"{name:<18}{p:>8.3f}{r:>8.3f}{ap50:>8.3f}{ap:>10.3f}  {'OK' if ok else 'LOW: add more frames of this product'}")
print(f"{'all':<18}{box.mp:>8.3f}{box.mr:>8.3f}{box.map50:>8.3f}{box.map:>10.3f}")
print()
print("PASS: every class mAP50 >= 0.90" if all_ok else "NOT YET: see TRAINING.md 'If mAP50 is low'")

In [ ]:
# 8. Optional: look at predictions on a few validation images
import random
val_dir = os.path.join(DATASET, data["val"])
samples = random.sample(sorted(os.listdir(val_dir)), k=min(4, len(os.listdir(val_dir))))
for r in best.predict([os.path.join(val_dir, s) for s in samples], imgsz=640, conf=0.55, verbose=False):
    display(Image(data=__import__("cv2").imencode(".jpg", r.plot())[1].tobytes(), width=640))

In [ ]:
# 9. Download the weights as speedmart_yolo.pt -> put it in the repo at models/speedmart_yolo.pt
shutil.copy(BEST, "/content/speedmart_yolo.pt")
files.download("/content/speedmart_yolo.pt")